<a href="https://colab.research.google.com/github/Orzilber/computationl-statistics/blob/collab/jackknife_and_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Jackknife Example

In [ ]:
import numpy as np

def jackknife_comprehensive_with_comparisons(n=1000, sigma_sq=4, batches=10000):
    np.random.seed(42)
    sigma = np.sqrt(sigma_sq)

    # Batched execution for memory efficiency: shape (batches, n)
    X = np.random.normal(0, sigma, (batches, n))

    # ==========================================
    # T1: The Raw Sum (Extensive)
    # ==========================================
    T1 = np.sum(X**2, axis=1)

    # LOO logic for T1
    t1_loo = T1[:, None] - X**2
    t1_loo_mean = np.mean(t1_loo, axis=1)

    bias_jack_t1 = (n - 1) * (t1_loo_mean - T1)
    v_jack_t1 = ((n - 1) / n) * np.sum((t1_loo - t1_loo_mean[:, None])**2, axis=1)
    v_theo_t1 = 2 * n * (sigma_sq**2)

    # ==========================================
    # T3: The Averaged Sum (Intensive Scaled Version of T1)
    # ==========================================
    T3 = np.mean(X**2, axis=1)

    # LOO logic for T3 (divide the remaining sum by n-1)
    t3_loo = (T1[:, None] - X**2) / (n - 1)
    t3_loo_mean = np.mean(t3_loo, axis=1)

    # Jackknife estimates for T3
    bias_jack_t3 = (n - 1) * (t3_loo_mean - T3)
    v_jack_t3 = ((n - 1) / n) * np.sum((t3_loo - t3_loo_mean[:, None])**2, axis=1)

    # Rescale T3 back to T1 scale
    rescaled_bias_t1 = n * bias_jack_t3
    rescaled_v_t1 = (n**2) * v_jack_t3

    # ==========================================
    # T2: Absolute value of the mean (|x_bar|)
    # ==========================================
    T2_sum = np.sum(X, axis=1)
    T2 = np.abs(T2_sum / n)

    # Vectorized LOO absolute mean across all batches
    x_bar_loo = (T2_sum[:, None] - X) / (n - 1)
    t2_loo = np.abs(x_bar_loo)
    t2_loo_mean = np.mean(t2_loo, axis=1)

    # Jackknife estimates
    bias_jack_t2 = (n - 1) * (t2_loo_mean - T2)
    v_jack_t2 = ((n - 1) / n) * np.sum((t2_loo - t2_loo_mean[:, None])**2, axis=1)

    # Theoretical Values
    v_theo_t2 = (sigma_sq / n) * (1 - (2 / np.pi))
    expected_t2_bias = sigma * np.sqrt(2 / (np.pi * n))

    # ==========================================
    # Output: Means over all batches
    # ==========================================
    print(f"            --- Configuration ---")
    print(f"Sample size (n) = {n}, Variance (sigma^2) = {sigma_sq}, Batches = {batches}\n")

    print("=== T1: The Extensive Trap (Raw Sum) ===")
    print(f"Mean Full Statistic (T_n):      {np.mean(T1):.2f}")
    print(f"Mean LOO Statistic (T_bar_n-1): {np.mean(t1_loo_mean):.2f}")
    print(f"-> Difference (T_bar_n-1 - T_n): {np.mean(t1_loo_mean - T1):.4f} (Causes the huge bias multiplier)")
    print(f"Theoretical Variance:           {v_theo_t1:.2f}")
    print(f"Mean Jackknife Variance:        {np.mean(v_jack_t1):.2f}")
    print(f"Mean Jackknife Bias:            {np.mean(bias_jack_t1):.2f} (Erroneously huge)\n")

    print("=== T3: The Averaged Sum (Intensive Scaled Version of T1) ===")
    print(f"Mean Full Statistic (T_n):      {np.mean(T3):.4f}")
    print(f"Mean LOO Statistic (T_bar_n-1): {np.mean(t3_loo_mean):.4f}")
    print(f"-> Difference (T_bar_n-1 - T_n): {np.mean(t3_loo_mean - T3):.4f} (Perfectly stable)")
    print(f"Theoretical Variance:           {v_theo_t1:.2f}")
    print(f"Mean Rescaled Variance:         {np.mean(rescaled_v_t1):.2f}")
    print(f"Mean Rescaled Bias:             {np.mean(rescaled_bias_t1):.2f} (Correctly unbiased)\n")

    print("=== T2: Absolute Mean ===")
    print(f"Mean Full Statistic (T_n):      {np.mean(T2):.6f}")
    print(f"Mean LOO Statistic (T_bar_n-1): {np.mean(t2_loo_mean):.6f}")
    print(f"-> Difference (T_bar_n-1 - T_n): {np.mean(t2_loo_mean - T2):.8f}")
    print(f"Theoretical True Bias of T2:    {expected_t2_bias:.6f}")
    print(f"Mean Jackknife Bias Estimate:   {np.mean(bias_jack_t2):.6f}")
    print(f"True Bias of V_jack(T2):        {np.mean(v_jack_t2) - v_theo_t2:.8f}")

# Execute the simulation
jackknife_comprehensive_with_comparisons()

            --- Configuration ---
Sample size (n) = 1000, Variance (sigma^2) = 4, Batches = 10000

=== T1: The Extensive Trap (Raw Sum) ===
Mean Full Statistic (T_n):      4000.11
Mean LOO Statistic (T_bar_n-1): 3996.11
-> Difference (T_bar_n-1 - T_n): -4.0001 (Causes the huge bias multiplier)
Theoretical Variance:           32000.00
Mean Jackknife Variance:        31915.92
Mean Jackknife Bias:            -3996.11 (Erroneously huge)

=== T3: The Averaged Sum (Intensive Scaled Version of T1) ===
Mean Full Statistic (T_n):      4.0001
Mean LOO Statistic (T_bar_n-1): 4.0001
-> Difference (T_bar_n-1 - T_n): 0.0000 (Perfectly stable)
Theoretical Variance:           32000.00
Mean Rescaled Variance:         31979.85
Mean Rescaled Bias:             0.00 (Correctly unbiased)

=== T2: Absolute Mean ===
Mean Full Statistic (T_n):      0.049664
Mean LOO Statistic (T_bar_n-1): 0.049689
-> Difference (T_bar_n-1 - T_n): 0.00002545
Theoretical True Bias of T2:    0.050463
Mean Jackknife Bias Estimate:

# Bootstrap Hypothesis Test

In [ ]:
import numpy as np
from scipy import stats

def run_f_test_suite(X, Y, scenario_name, batches=100000):
    n, m = len(X), len(Y)

    # Calculate Observed Statistic
    var_X = np.var(X, ddof=1)
    var_Y = np.var(Y, ddof=1)
    F_obs = var_X / var_Y

    # ==========================================
    # 1. Analytical Parametric F-Test
    # ==========================================
    p_analytical = 2 * min(stats.f.cdf(F_obs, n-1, m-1), stats.f.sf(F_obs, n-1, m-1))

    # ==========================================
    # 2. Setup H0 (Direct Scaling without Centering)
    # ==========================================
    # Divide directly by the standard deviation to force Variance = 1
    X_null = X / np.sqrt(var_X)
    Y_null = Y / np.sqrt(var_Y)

    # ==========================================
    # 3. Parametric Bootstrap F-Test
    # ==========================================
    # Draw from Normal distributions with Var=1 and matching scaled means
    X_par = np.random.normal(np.mean(X_null), 1.0, size=(batches, n))
    Y_par = np.random.normal(np.mean(Y_null), 1.0, size=(batches, m))

    F_par = np.var(X_par, axis=1, ddof=1) / np.var(Y_par, axis=1, ddof=1)
    p_par_boot = 2 * min(np.mean(F_par >= F_obs), np.mean(F_par <= F_obs))

    # ==========================================
    # 4. Non-Parametric Empirical Bootstrap F-Test
    # ==========================================
    # Resample with replacement from the uncentered, Var=1 empirical data
    idx_X = np.random.randint(0, n, size=(batches, n))
    idx_Y = np.random.randint(0, m, size=(batches, m))

    X_emp = X_null[idx_X]
    Y_emp = Y_null[idx_Y]

    F_emp = np.var(X_emp, axis=1, ddof=1) / np.var(Y_emp, axis=1, ddof=1)
    p_emp_boot = 2 * min(np.mean(F_emp >= F_obs), np.mean(F_emp <= F_obs))

    # ==========================================
    # Output Results
    # ==========================================
    print(f"=== {scenario_name} ===")
    print(f"Observed F-statistic: {F_obs:.4f}\n")

    print(f"1. Analytical F-Test:        p-value = {p_analytical:.6f}  -> {'REJECT H0 (False Positive!)' if p_analytical < 0.05 else 'Fail to Reject H0 (Correct)'}")
    print(f"2. Parametric Bootstrap:     p-value = {p_par_boot:.6f}  -> {'REJECT H0 (False Positive!)' if p_par_boot < 0.05 else 'Fail to Reject H0 (Correct)'}")
    print(f"3. Non-Parametric Bootstrap: p-value = {p_emp_boot:.6f}  -> {'REJECT H0 (False Positive!)' if p_emp_boot < 0.05 else 'Fail to Reject H0 (Correct)'}\n")

def comprehensive_market_simulation():
    n_days = 500

    # ---------------------------------------------------------
    # SCENARIO A: The "Textbook" Normal Market
    # Both indices follow perfect Normal distributions.
    # True Variance is identical (H0 is TRUE).
    # ---------------------------------------------------------
    np.random.seed(42)
    sp500_normal = np.random.normal(0, np.sqrt(2.0), size=n_days)
    nasdaq_normal = np.random.normal(0, np.sqrt(2.0), size=n_days)

    run_f_test_suite(
        sp500_normal, nasdaq_normal,
        "Scenario A: The Textbook Normal Market (Where Bootstrap is Unnecessary)"
    )

    print("-" * 70 + "\n")

    # ---------------------------------------------------------
    # SCENARIO B: The "Black Swan" Market
    # Both indices follow identical t-distributions (df=2.1).
    # True Variance is identical (H0 is unequivocally TRUE).
    # We search for a natural realization where F_obs > 1.35.
    # ---------------------------------------------------------
    seed = 0
    while True:
        np.random.seed(seed)
        sp500_extreme = np.random.standard_t(df=2.1, size=n_days)
        nasdaq_extreme = np.random.standard_t(df=2.1, size=n_days)

        var_s = np.var(sp500_extreme, ddof=1)
        var_n = np.var(nasdaq_extreme, ddof=1)
        F_obs = var_s / var_n

        # Stop when we find a naturally occurring gap of ~35%
        if 1.35 < F_obs < 1.40:
            break
        seed += 1

    run_f_test_suite(
        sp500_extreme, nasdaq_extreme,
        "Scenario B: The Black Swan Market (Extreme Fat Tails)"
    )

# Execute the full comparison
comprehensive_market_simulation()

=== Scenario A: The Textbook Normal Market (Where Bootstrap is Unnecessary) ===
Observed F-statistic: 1.0067

1. Analytical F-Test:        p-value = 0.940843  -> Fail to Reject H0 (Correct)
2. Parametric Bootstrap:     p-value = 0.939360  -> Fail to Reject H0 (Correct)
3. Non-Parametric Bootstrap: p-value = 0.937440  -> Fail to Reject H0 (Correct)

----------------------------------------------------------------------

=== Scenario B: The Black Swan Market (Extreme Fat Tails) ===
Observed F-statistic: 1.3528

1. Analytical F-Test:        p-value = 0.000761  -> REJECT H0 (False Positive!)
2. Parametric Bootstrap:     p-value = 0.000780  -> REJECT H0 (False Positive!)
3. Non-Parametric Bootstrap: p-value = 0.243420  -> Fail to Reject H0 (Correct)

